# 策略与奖励

本节介绍 RL 中最基础的两个概念：策略（Policy）和奖励（Reward）。它们是所有 RL 算法的基石。

---

## 1. 策略（Policy）

在 RL 中，**策略**就是模型本身——给定输入，模型输出一个概率分布，表示它选择每个可能输出的概率。

对于语言模型来说，策略就是模型在每一步生成 token 的概率分布：

<img src="./images/policy_distribution.png" alt="策略从 prompt 到采样输出" width="90%">

在 RL 中，策略就是当前语言模型。给定一个 prompt 后，策略会输出 token 概率分布；采样时并不是固定选择唯一答案，而是按概率从分布中选出下一个 token。RL 训练要做的事情，就是调整这个分布，让高奖励输出的概率逐渐上升。

策略的好坏取决于它是否能产生高奖励的输出。RL 训练的目标就是**优化策略**，让模型更倾向于产生高奖励的输出。

### 策略的两种表示

在 RL 训练中，我们会遇到两个策略：

| 策略 | 符号 | 说明 |
|------|------|------|
| **当前策略** | π | 正在训练的模型，每个 step 都在更新 |
| **参考策略** | π_ref | 训练前的初始模型（SFT 产物），固定不变 |

为什么要保留参考策略？因为 RL 训练容易让模型「跑偏」——为了获得高奖励而忘记原有的语言能力。参考策略的作用是**拉住当前策略**，防止它偏离太远。这个机制叫做 KL 正则化，我们在 2.4 节会详细介绍。

---

## 2. 奖励（Reward）

**奖励**是奖励函数对模型输出的评分。在 Wordle RL 中，奖励由 `wordle_reward.py` 计算：

<img src="./images/reward_components.png" alt="Wordle 奖励组成" width="90%">

Wordle 的总奖励由多个信号组成：

- `correct_answer`：猜中秘密单词，提供稀疏但最重要的成功信号。
- `partial_answer`：根据绿色、黄色反馈给分，提供更密集的中间信号。
- `length_bonus`：鼓励更少步数猜中，避免模型拖长过程。
- `format_reward`：鼓励使用规范格式 `<guess>[word]</guess>`，但权重较低，防止模型只学格式。

### 奖励函数设计原则

设计好的奖励函数是 RL 训练成功的关键。Wordle 的奖励函数遵循以下原则：

1. **稀疏 + 密集结合**：`correct_answer` 是稀疏奖励（只有猜中才有），`partial_answer` 是密集奖励（每轮都有），两者结合加速学习。
2. **分层设计**：完全猜中 > 部分匹配 > 格式正确，引导模型循序渐进地提升。
3. **防 reward hacking**：`format_reward` 权重很低（0.2），防止模型只学格式不学猜词。

---

## 3. Rollout

**Rollout** 是 RL 中的核心概念：让当前策略生成一条完整的回复。在 Wordle 中，一个 rollout 就是模型完成一局完整的猜词游戏（多轮交互）。

GRPO 的关键思想是对**同一个 prompt 生成多条 rollout**，然后比较它们的奖励：

```text
同一个 prompt 生成 8 条 rollout (rollout.n=8)

prompt: 猜一个5字母单词，答案是 crane

rollout 1: guess1=apple -> guess2=crane -> 猜中!  reward=1.5
rollout 2: guess1=house -> guess2=world -> ... -> 未猜中  reward=0.6
rollout 3: guess1=crane -> 猜中!  reward=1.8
...
rollout 8: guess1=zzzzz -> 格式错误  reward=0.2

比较这 8 条 rollout 的奖励，强化好的、抑制差的
```

这就是 RL 的试错学习方式：不提供逐 token 的标准回复，只给出用于自动评分的秘密单词 `crane`。模型尝试多条轨迹后，GRPO 会提高高优势轨迹的概率。

---

## 课后练习

1. （判断题）在 RL 训练中，策略就是语言模型本身，它给定输入后输出一个 token 概率分布。

2. （判断题）参考策略（π_ref）在每个训练 step 都会更新，与当前策略保持同步。

3. （判断题）Wordle 的奖励函数中，correct_answer 是稀疏奖励，partial_answer 是密集奖励。

4. （单选题）保留参考策略的主要目的是什么？
    A. 加速训练
    B. 防止当前策略偏离初始模型太远
    C. 计算 reward
    D. 生成 rollout

5. （单选题）GRPO 中对同一个 prompt 生成多条 rollout 的主要目的是什么？
    A. 增加训练数据量
    B. 比较不同 rollout 的奖励，计算相对优势
    C. 提高推理速度
    D. 减少 KL 散度

In [ ]:
!cat ../cann-learning-hub/tutorials/rl_training_pipeline/02_rl_core_concepts/answer/02.02_answer.txt